In [ ]:
# Cell 0: GPU Hardware Check & Setup
# This cell detects your GPU and estimates whether mt5-large will fit
import torch
import sys
from IPython.display import display, HTML

def check_gpu():
    """Check GPU availability and memory, estimate if mt5-large fits."""
    if not torch.cuda.is_available():
        display(HTML("""
        <div style="background: #ffe6e6; padding: 10px; border-radius: 5px; border: 1px solid #ffcccc;">
            <strong>WARNING: No GPU detected!</strong><br>
            Training mt5-large on CPU will take ~2 days and likely OOM.<br>
            Please use a GPU (Colab T4/V100/A100, or local 16GB+ GPU).
        </div>
        """))
        return False, None, None
    
    device_name = torch.cuda.get_device_name(0)
    device_cap = torch.cuda.get_device_capability(0)
    total_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    # Estimate memory needs for mt5-large
    # Model weights: ~4.6GB (fp32) / ~2.3GB (bf16)
    # Gradients: ~2.3GB (bf16)  
    # Optimizer state: AdamW ~19GB, Adafactor <1GB
    # Activations (gradient checkpointing): ~4-6GB
    # Total with AdamW: ~30GB+, with Adafactor: ~12GB
    
    adafactor_fits = total_memory_gb >= 12  # Conservative estimate
    adamw_fits = total_memory_gb >= 24  # Very conservative
    
    display(HTML(f"""
    <div style="background: #e6f7ff; padding: 10px; border-radius: 5px; border: 1px solid #b3d7ff;">
        <strong>GPU detected:</strong> {device_name}<br>
        <strong>Memory:</strong> {total_memory_gb:.1f} GB<br>
        <strong>Capability:</strong> {device_cap}<br>
        <strong>bf16 support:</strong> {torch.cuda.is_bf16_supported()}<br><br>
        <strong>Memory fit estimate:</strong><br>
        &bull; Adafactor optimizer (recommended): {'FITS' if adafactor_fits else 'TOO SMALL'} (needs ~12GB)<br>
        &bull; AdamW optimizer: {'FITS' if adamw_fits else 'TOO SMALL'} (needs ~24GB)<br>
    </div>
    """))
    
    if not adafactor_fits:
        display(HTML("""
        <div style="background: #fff3cd; padding: 10px; border-radius: 5px; border: 1px solid #ffc107;">
            <strong>WARNING: GPU memory too small for mt5-large!</strong><br>
            Consider: mt5-base (fits in ~8GB) or use a larger GPU.<br>
            The notebook will use configs/mt5_base.yaml instead.
        </div>
        """))
    
    return True, total_memory_gb, adafactor_fits

has_gpu, memory_gb, fits_large = check_gpu()
print(f"Python {sys.version}")
print(f"PyTorch {torch.__version__}")

# Multilingual Health QA — GPU Training Notebook

This notebook trains a competition-grade mT5-large model for the Zindi MSRH challenge. It's designed to run on Google Colab (free T4 or paid GPU) or any local GPU with 16GB+ VRAM.

## Key Features
- **Memory-efficient**: Uses Adafactor optimizer to fit mt5-large in 16GB GPU
- **Robust loading**: Handles torch version compatibility issues
- **Complete pipeline**: EDA → Training → Evaluation → Submission
- **Colab-optimized**: Auto-detects GPU, handles file paths

## 1 — Environment & data

In [ ]:
# Cell 1: Environment & Data Setup
# On Colab, clone the repo and install deps. Locally, skip the clone.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("MultiHealthQA"):
        # Clone your repo here - UPDATE WITH YOUR REPO URL
        subprocess.run(["git", "clone", "https://github.com/kelvintawe12/MultiHealthQA.git"], check=True)
    os.chdir("MultiHealthQA")
    print("Installing dependencies...")
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
    print("Dependencies installed")
else:
    print("Running locally - skipping repo clone")

# Make the package importable.
sys.path.insert(0, os.path.abspath("src"))
print(f"Working directory: {os.getcwd()}")

# Verify data files exist
from pathlib import Path
print("Checking data files:")
data_ok = True
for f in ["Train.csv", "Val.csv", "Test.csv", "SampleSubmission.csv"]:
    p = Path("data") / f
    exists = p.exists()
    print(f"  {'OK' if exists else 'MISSING'} {f}")
    if not exists:
        data_ok = False

if not data_ok:
    print("WARNING: Data files missing! Please upload them to the data/ directory.")
    if IN_COLAB:
        print("On Colab: Use the folder icon on the left to upload files to data/")

In [ ]:
# The 5 competition CSVs must live in ./data (Train/Val/Test/SampleSubmission).
# On Colab, upload them or mount Drive; locally they are already there.
from pathlib import Path
for f in ["Train.csv", "Val.csv", "Test.csv", "SampleSubmission.csv"]:
    p = Path("data") / f
    print(("✅" if p.exists() else "❌"), p)

In [ ]:
from mhqa.data import load_all
data = load_all("data")
train, val, test = data["train"], data["val"], data["test"]
print("train", train.shape, "| val", val.shape, "| test", test.shape)
train.head(3)

## 2 — Exploratory Data Analysis

The decisive findings (regenerated by `python -m scripts.run_eda`):

* **Canonical templated answers** — only ~61% of answers are unique; the most
  common answer recurs 68×. The task rewards reproducing canonical, in-language
  health responses → high ROUGE is achievable, and **fine-tuned generation** is
  the right tool.
* **Retrieval ceiling ≈ R1 0.44** (per-language char-ngram NN on a CV holdout) —
  below target, so retrieval is a *fallback*, not the model.
* **Mixed scripts** — Amharic = Ge'ez (≈2% Latin), Akan = Latin+diacritics. We
  never lowercase or strip non-ASCII.
* **Answer length** — p90 = 153 words, max 482 → `max_target_length = 512`.


In [ ]:
# Regenerate all EDA stats + figures (writes reports/figures/*.png).
import subprocess, sys
subprocess.run([sys.executable, "-m", "scripts.run_eda", "--data-dir", "data", "--out", "reports"], check=True)

In [ ]:
from IPython.display import Image, display
for fig in ["01_subset_distribution", "02_length_distributions",
            "03_answer_templating", "04_script_composition"]:
    p = f"reports/figures/{fig}.png"
    print(p)
    display(Image(p))

## 3 — Baselines

In [ ]:
# Baseline A — retrieval floor (Experiment 1). Fast, CPU-only.
from mhqa.data import stratified_split, ANSWER_COL, LANG_COL
from mhqa.retrieval import PerLanguageRetriever
from mhqa.metrics import compute_rouge, compute_rouge_by_language

trn, holdout = stratified_split(train, val_size=0.05, seed=42)
retr = PerLanguageRetriever().fit(trn)
preds, _, _ = retr.predict(holdout)
print("Retrieval baseline:", compute_rouge(preds, holdout[ANSWER_COL].tolist()))
compute_rouge_by_language(preds, holdout[ANSWER_COL].tolist(), holdout[LANG_COL].tolist()).round(4)

## 4 — Fine-tune mT5

`configs/mt5_base.yaml` is the safe default (Colab T4). On a 16–24 GB GPU switch
to `configs/mt5_large.yaml` for the best-score run that targets the leaderboard.


In [ ]:
from mhqa.config import load_config
cfg = load_config("configs/mt5_base.yaml")
# For a fast smoke run on weak hardware, uncomment:
# cfg = cfg.with_overrides(model_name="google/mt5-small", num_train_epochs=1)
cfg.model_name, cfg.num_train_epochs, cfg.max_target_length

In [ ]:
# Cell 4: Select Config & Start Training
# Automatically select config based on GPU memory detected in cell 0
from mhqa.config import load_config

# Auto-select config based on hardware check
if has_gpu and fits_large:
    config_path = "configs/mt5_large.yaml"
    print(f"Using {config_path} (Adafactor optimizer for 16GB GPU)")
else:
    config_path = "configs/mt5_base.yaml" 
    print(f"Using {config_path} (safe default for smaller GPUs/Colab T4)")

cfg = load_config(config_path)
print(f"\nConfig summary:")
print(f"  Model: {cfg.model_name}")
print(f"  Optimizer: {cfg.optimizer}")
print(f"  Epochs: {cfg.num_train_epochs}")
print(f"  Batch size: {cfg.per_device_train_batch_size} x {cfg.gradient_accumulation_steps} = {cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps} effective")
print(f"  Max length: {cfg.max_input_length} / {cfg.max_target_length}")
print(f"  Precision: {'bf16' if cfg.bf16_if_supported else 'fp16/fp32'}")

## 5 — Held-out evaluation (overall + per-language)

In [ ]:
from mhqa.evaluate import evaluate_model, holdout_for
holdout = holdout_for(cfg)
overall, per_lang, _ = evaluate_model(trainer.model, tokenizer, holdout, cfg, batch_size=8)
print(f"ROUGE-1={overall['rouge1_f1']:.4f}  ROUGE-L={overall['rougeL_f1']:.4f}  "
      f"combined={overall['combined']:.4f}")
per_lang.round(4)

## 6 — Experiment suite

Each experiment is a config delta + hypothesis, logged to
`reports/experiments.csv`. Run individually or all at once (long — GPU box).


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "scripts.run_experiments", "--list"], check=True)
# Example single run:
# subprocess.run([sys.executable, "-m", "scripts.run_experiments",
#                 "--config", "configs/mt5_base.yaml",
#                 "--only", "exp04_mt5base_langprefix"], check=True)

In [ ]:
import pandas as pd, os
if os.path.exists("reports/experiments.csv"):
    display(pd.read_csv("reports/experiments.csv"))

## 7 — Generate the validated test submission

In [ ]:
from mhqa.data import ID_COL, load_split
from mhqa.infer import predict_dataframe
from mhqa.submit import make_submission

retriever = None
if cfg.retrieval_augment or cfg.hybrid_fallback:
    retriever = PerLanguageRetriever().fit(load_split(cfg.train_path, has_answer=True))

test_df = load_split(cfg.test_path, has_answer=False)
test_preds = predict_dataframe(trainer.model, tokenizer, test_df, cfg,
                               retriever=retriever, batch_size=8)
sub = make_submission(
    ids=test_df[ID_COL].tolist(),
    predictions=test_preds,
    output_path=cfg.submission_path,
    sample_submission_path=cfg.sample_submission_path,  # validates shape + IDs
)
print("Wrote", cfg.submission_path)
sub.head()

## Next steps to climb the leaderboard

1. Re-run section 4 with `configs/mt5_large.yaml` on the GPU.
2. Sweep `exp06`→`exp11` and record `public_lb` in `reports/experiments.csv`.
3. Update `reports/leaderboard_progression.csv` after each submission and plot it
   for the report's progression chart.
